# 十國藥價下載（Colab 版）

**這份 notebook 只做資料下載，不做佐證截圖。**

佐證截圖請在自己的電腦上執行。原因是截圖要作為爭議時的佐證，
需要能說清楚是何時何地由誰擷取；從雲端機房擷取會讓這件事變弱，
而且可能因地區而看到不同的頁面內容。詳見 `docs/Colab評估.md`。

---

> ⚠️ **使用前請先確認**
>
> 把藥品清單上傳到 Colab，等於把它交給 Google 的雲端服務處理。
> 那份清單等於揭露正在調查哪些藥品的國際價格，具市場敏感性。
> **請先與主管及資安單位確認是否符合單位規定再使用。**

---

## 什麼時候值得用這份 notebook

- 公司網路擋掉某些官方網站，本機連不到
- 想先看流程跑起來，還沒處理好本機環境
- 開發階段要試跑比較吃資源的解析（大型 PDF、XML）

其餘情況建議直接在本機執行 `開啟藥價工具.bat`，比較單純。


## 第 1 步：取得程式

從 GitHub 取得。若儲存庫是私有的，改用下方的「手動上傳」那一格。

In [ ]:
# 從 GitHub 取得程式
REPO = "https://github.com/remmus38180-cpu/-.git"
BRANCH = "claude/ten-country-drug-price-automation-fyq8fb"

import os, shutil, subprocess
if os.path.isdir("drugprice-repo"):
    shutil.rmtree("drugprice-repo")
result = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "drugprice-repo"],
    capture_output=True, text=True)
print(result.stdout or result.stderr)

if os.path.isdir("drugprice-repo/drugprice"):
    os.chdir("drugprice-repo")
    print("已取得程式，目前目錄：", os.getcwd())
else:
    print("取得失敗。若儲存庫是私有的，請改用下一格手動上傳。")


In [ ]:
# 手動上傳（只有在上面那格失敗時才需要執行）
# 請先在本機把整個 drugprice 資料夾壓縮成 drugprice.zip
from google.colab import files       # type: ignore
import zipfile, os

uploaded = files.upload()
for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as zf:
            zf.extractall(".")
        print("已解壓縮", name)
print("目前目錄內容：", os.listdir("."))


## 第 2 步：確認環境

本專案只用 Python 標準函式庫，Colab 不需要安裝任何套件。

In [ ]:
import sys
sys.path.insert(0, ".")

from drugprice.countries import COUNTRIES, METHOD_LABELS
from drugprice.registry import codes

print("Python", sys.version.split()[0])
print("可自動下載的國家：", "、".join(codes()))
print()
print("十國執行方式：")
for country in COUNTRIES:
    print(f"  {country.code:6} {country.name:14} {METHOD_LABELS[country.method]}")


## 第 3 步：藥品清單

直接在下面編輯，或改用上傳檔案。只有商品名也可以，但**建議一併填成分名**，跨國商品名差異很大。

In [ ]:
藥品清單 = """商品名,成分名,ATC,含量,藥商
Eliquis,apixaban,B01AF02,2.5 mg,Bristol-Myers Squibb
Zelboraf,vemurafenib,L01EC01,240 mg,Roche
"""

with open("藥品清單.csv", "w", encoding="utf-8") as fh:
    fh.write(藥品清單)

from drugprice.matching.druglist import read_drug_list
queries, messages = read_drug_list(藥品清單)
for message in messages:
    print(message)


## 第 4 步：下載

澳洲的官方 API 規定每 20 秒 1 次請求，且此上限由全世界的使用者共用，因此澳洲一國約需 2 分鐘。**這是伺服器端的限制，在 Colab 上跑一樣要等。**

In [ ]:
要下載的國家 = ["JP", "FR", "BE", "SE", "AU", "CA-SK"]

import os
from drugprice.core.net import Fetcher
from drugprice.core.rawstore import RawStore
from drugprice.core.schema import write_csv
from drugprice.registry import get
from drugprice.sources.base import Context

store = RawStore("raw")
ctx = Context(Fetcher(min_interval=1.0, log=print), store, log=print)
os.makedirs("output", exist_ok=True)

失敗 = []
for code in 要下載的國家:
    try:
        records, _ = get(code).run(ctx)
        path = os.path.join("output", f"{code}_藥價.csv")
        write_csv(records, path)
        print(f"　已輸出 {path}（{len(records):,} 筆）\n")
    except Exception as exc:
        print(f"[{code}] 失敗：{exc}\n")
        失敗.append(code)

print("=" * 60)
print(f"完成 {len(要下載的國家) - len(失敗)} 國" + (f"，失敗：{'、'.join(失敗)}" if 失敗 else ""))


## 第 5 步：比對與排序

排序不是決定。所有候選都會列出，最終由人判斷。每一筆的「排序依據」會寫明分數是怎麼來的。

In [ ]:
比對國家 = "FR"

from drugprice.core.schema import read_csv
from drugprice.matching.matcher import match_country

records = read_csv(f"output/{比對國家}_藥價.csv")
for result in match_country(queries, records, limit_per_query=3):
    print("=" * 70)
    print(f"{result.query.brand_name}（{result.query.generic_name}）")
    if result.message:
        print("  ", result.message)
    for candidate in result.candidates:
        cheapest = candidate.cheapest()
        print(f"\n  [第 {candidate.rank} 順位] {candidate.tier} {candidate.tier_label}"
              f"　{candidate.brand_name}")
        print(f"    原廠註記：{candidate.originator_flag or '（無）'}"
              f"　藥商：{candidate.marketer}")
        if cheapest:
            print(f"    {cheapest.price} {cheapest.currency}"
                  f"［{cheapest.category_label}］單價 {cheapest.unit_price}")
        print(f"    排序依據：{candidate.ranking_basis}")
        if candidate.caution:
            print(f"    ⚠ {candidate.caution}")


## 第 6 步：核對原始檔雜湊

**這一步是佐證鏈的關鍵。** 原始檔的 SHA-256 會一起打包帶回本機，回到本機後可以再算一次核對，確認檔案在傳輸過程中沒有被更動。佐證鏈不會斷在雲端。

In [ ]:
import csv

問題 = store.verify()
if 問題:
    print("以下檔案有問題：")
    for name, reason in 問題:
        print(f"  {name}：{reason}")
else:
    print("留存區內所有原始檔的雜湊都正確。\n")

with open(store.manifest_path, encoding="utf-8-sig", newline="") as fh:
    for row in csv.DictReader(fh):
        print(f"{row['國家']:6} {row['檔案']:34} {int(row['位元組']):>10,} 位元組")
        print(f"       SHA-256 {row['SHA-256']}")
        print(f"       來源 {row['來源網址']}")


## 第 7 步：打包帶回本機

把原始檔、轉換結果與雜湊清單一起打包下載。回到本機解壓縮後，可以直接用 `開啟藥價工具.bat` 接著做複核與佐證。

In [ ]:
import shutil
from datetime import datetime, timedelta, timezone

台北 = timezone(timedelta(hours=8))
檔名 = "十國藥價_" + datetime.now(台北).strftime("%Y%m%d_%H%M")

shutil.make_archive(檔名, "zip", ".", "raw")
shutil.make_archive(檔名 + "_output", "zip", ".", "output")

from google.colab import files       # type: ignore
files.download(檔名 + ".zip")
files.download(檔名 + "_output.zip")
print("已開始下載。回到本機後解壓縮到工具的資料夾即可。")


---

## 不要在 Colab 做的事

| 不要做 | 原因 |
|---|---|
| 佐證截圖 | 需要可追溯性；雲端機房的 IP 可能看到不同內容，且執行環境不在單位掌控下 |
| 存放 API 金鑰 | 英國 TRUD 的金鑰屬個人憑證，不要留在雲端 notebook 裡 |
| 長期排程 | Colab 執行階段會被回收，不適合當固定作業的基礎設施 |
| 改用第三方套件 | 成品要能在同仁的 WinPython 免安裝版執行，只能用標準函式庫 |
